# Evaluating AI Models on Maritime Data Analysis

**Author:** Anointed Akai

### The question
Can a small open-source AI model be trusted to answer business questions about a shipping fleet by writing its own Pandas code? And does a better-written prompt make it more reliable?

### What this notebook does
1. Loads a public Kaggle dataset of 2,736 ship voyage records.
2. Builds a **benchmark** of 20 business questions, each with a reference solution I wrote and verified.
3. Asks two AI models to answer every question by writing Pandas code, under two different prompts.
4. **Runs and grades** each response automatically against the reference answer.
5. **Ranks** the models, classifies their failures, and records a written rationale for each failure.
6. Exports the verified question-and-solution pairs as a **fine-tuning dataset** (JSONL).

### Skills shown
Python, Pandas, benchmark design, model evaluation (evals), prompt engineering, error analysis, and supervised fine-tuning (SFT) data preparation.

## 1. Setup

Run this in Google Colab with a **T4 GPU** (Runtime > Change runtime type > T4 GPU). It also runs on CPU, only slower.

`DRY_RUN = True` swaps the real models for a simulated one so the whole pipeline can be tested in seconds. **Results from a dry run are fake and must never be reported.**

In [ ]:
DRY_RUN = False          # True = test the pipeline with a simulated model (results are NOT real)
DATA_PATH = None         # set to a CSV path to skip the Kaggle download

KAGGLE_DATASET = "jeleeladekunlefijabi/ship-performance-clustering-dataset"

MODELS = {
    "Qwen2.5-Coder-0.5B": "Qwen/Qwen2.5-Coder-0.5B-Instruct",
    "Qwen2.5-Coder-1.5B": "Qwen/Qwen2.5-Coder-1.5B-Instruct",
}

In [ ]:
import builtins, contextlib, glob, io, json, math, os, random, re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 120)

## 2. Load and clean the data

The dataset describes voyages by four ship types in the Gulf of Guinea: speed, engine power, distance, cargo, cost, revenue and turnaround time.

In [ ]:
def load_data():
    if DATA_PATH:
        path = DATA_PATH
    else:
        import kagglehub
        folder = kagglehub.dataset_download(KAGGLE_DATASET)
        path = glob.glob(os.path.join(folder, "**", "*.csv"), recursive=True)[0]
    return pd.read_csv(path)

raw = load_data()
print(raw.shape)
raw.head()

In [ ]:
CATEGORICAL = ["Ship_Type", "Route_Type", "Engine_Type", "Maintenance_Status", "Weather_Condition"]

print("Missing values per column:")
print(raw[CATEGORICAL].replace("None", np.nan).isna().sum())

**Cleaning decisions**

- Each categorical column has missing labels. I label them `"Unknown"` instead of dropping the rows, because the numeric measurements in those rows are still valid.
- `Date` is converted to a real date so that monthly and quarterly questions are possible.

Every question below is asked against this cleaned table, so the model and the reference solutions work from identical data.

In [ ]:
df = raw.copy()
df["Date"] = pd.to_datetime(df["Date"])
for col in CATEGORICAL:
    df[col] = df[col].replace("None", np.nan).fillna("Unknown").astype(object)

NUMERIC = [c for c in df.columns if c not in CATEGORICAL + ["Date"]]
print(f"{len(df):,} rows, {df['Date'].min():%b %Y} to {df['Date'].max():%b %Y}")
df[CATEGORICAL].nunique()

## 3. The benchmark: 20 business questions

Each question has:

- a **difficulty** (easy = one step, medium = filter or derive then aggregate, hard = several steps),
- an **answer type** (number, text or list), which decides how it is graded,
- a **reference solution** in Pandas, written by me. Running it produces the ground truth.

Writing the reference as code, not as a typed-in number, means the ground truth can be re-checked by anyone.

In [ ]:
QUESTIONS = [
    # ---------- easy ----------
    dict(id="Q01", difficulty="easy", kind="number",
         question="How many records are for Tanker ships?",
         solution="answer = int((df['Ship_Type'] == 'Tanker').sum())"),
    dict(id="Q02", difficulty="easy", kind="number",
         question="What is the average operational cost in USD across all records?",
         solution="answer = float(df['Operational_Cost_USD'].mean())"),
    dict(id="Q03", difficulty="easy", kind="text",
         question="Which ship type has the highest average revenue per voyage? Exclude records where the ship type is Unknown.",
         solution="known = df[df['Ship_Type'] != 'Unknown']\nanswer = known.groupby('Ship_Type')['Revenue_per_Voyage_USD'].mean().idxmax()"),
    dict(id="Q04", difficulty="easy", kind="number",
         question="What is the total distance travelled, in nautical miles, by ships with a Diesel engine?",
         solution="answer = float(df.loc[df['Engine_Type'] == 'Diesel', 'Distance_Traveled_nm'].sum())"),
    dict(id="Q05", difficulty="easy", kind="number",
         question="What percentage of all records have a maintenance status of Critical?",
         solution="answer = float((df['Maintenance_Status'] == 'Critical').mean() * 100)"),
    dict(id="Q06", difficulty="easy", kind="number",
         question="What is the median turnaround time in hours for voyages in Rough weather?",
         solution="answer = float(df.loc[df['Weather_Condition'] == 'Rough', 'Turnaround_Time_hours'].median())"),
    # ---------- medium ----------
    dict(id="Q07", difficulty="medium", kind="text",
         question="Profit is revenue minus operational cost. Which route type has the highest average profit? Exclude records where the route type is Unknown.",
         solution="d = df[df['Route_Type'] != 'Unknown'].copy()\nd['profit'] = d['Revenue_per_Voyage_USD'] - d['Operational_Cost_USD']\nanswer = d.groupby('Route_Type')['profit'].mean().idxmax()"),
    dict(id="Q08", difficulty="medium", kind="list",
         question="Rank the engine types by average efficiency (nautical miles per kWh), from most efficient to least efficient. Exclude Unknown.",
         solution="d = df[df['Engine_Type'] != 'Unknown']\nanswer = d.groupby('Engine_Type')['Efficiency_nm_per_kWh'].mean().sort_values(ascending=False).index.tolist()"),
    dict(id="Q09", difficulty="medium", kind="number",
         question="How many records are loss-making, meaning operational cost is greater than revenue?",
         solution="answer = int((df['Operational_Cost_USD'] > df['Revenue_per_Voyage_USD']).sum())"),
    dict(id="Q10", difficulty="medium", kind="text",
         question="Which calendar month had the highest total revenue? Give the answer as text in the form YYYY-MM.",
         solution="monthly = df.groupby(df['Date'].dt.to_period('M'))['Revenue_per_Voyage_USD'].sum()\nanswer = str(monthly.idxmax())"),
    dict(id="Q11", difficulty="medium", kind="number",
         question="What is the average speed over ground, in knots, of Container Ships on Transoceanic routes?",
         solution="m = (df['Ship_Type'] == 'Container Ship') & (df['Route_Type'] == 'Transoceanic')\nanswer = float(df.loc[m, 'Speed_Over_Ground_knots'].mean())"),
    dict(id="Q12", difficulty="medium", kind="number",
         question="What is the Pearson correlation between engine power (kW) and speed over ground (knots)?",
         solution="answer = float(df['Engine_Power_kW'].corr(df['Speed_Over_Ground_knots']))"),
    dict(id="Q13", difficulty="medium", kind="number",
         question="What is the difference in average revenue per voyage between records with Good maintenance status and records with Critical maintenance status (Good minus Critical)?",
         solution="g = df.groupby('Maintenance_Status')['Revenue_per_Voyage_USD'].mean()\nanswer = float(g['Good'] - g['Critical'])"),
    dict(id="Q14", difficulty="medium", kind="number",
         question="What is the cost per nautical mile for Fish Carriers, calculated as total operational cost divided by total distance travelled?",
         solution="d = df[df['Ship_Type'] == 'Fish Carrier']\nanswer = float(d['Operational_Cost_USD'].sum() / d['Distance_Traveled_nm'].sum())"),
    # ---------- hard ----------
    dict(id="Q15", difficulty="hard", kind="text",
         question="Among records with Critical maintenance status, which ship type has the lowest average efficiency (nautical miles per kWh)? Exclude records where the ship type is Unknown.",
         solution="d = df[(df['Maintenance_Status'] == 'Critical') & (df['Ship_Type'] != 'Unknown')]\nanswer = d.groupby('Ship_Type')['Efficiency_nm_per_kWh'].mean().idxmin()"),
    dict(id="Q16", difficulty="hard", kind="number",
         question="What is the profit margin for Tankers as a percentage, calculated as total profit divided by total revenue times 100? Profit is revenue minus operational cost.",
         solution="d = df[df['Ship_Type'] == 'Tanker']\nrev = d['Revenue_per_Voyage_USD'].sum()\nanswer = float((rev - d['Operational_Cost_USD'].sum()) / rev * 100)"),
    dict(id="Q17", difficulty="hard", kind="text",
         question="Which combination of ship type and route type has the highest average profit (revenue minus operational cost)? Exclude any record where either value is Unknown. Give the answer as text in the form 'Ship_Type | Route_Type'.",
         solution="d = df[(df['Ship_Type'] != 'Unknown') & (df['Route_Type'] != 'Unknown')].copy()\nd['profit'] = d['Revenue_per_Voyage_USD'] - d['Operational_Cost_USD']\nship, route = d.groupby(['Ship_Type', 'Route_Type'])['profit'].mean().idxmax()\nanswer = f'{ship} | {route}'"),
    dict(id="Q18", difficulty="hard", kind="number",
         question="On how many distinct dates was the total operational cost of all records on that date greater than half of the total revenue on that date?",
         solution="daily = df.groupby('Date')[['Operational_Cost_USD', 'Revenue_per_Voyage_USD']].sum()\nanswer = int((daily['Operational_Cost_USD'] > 0.5 * daily['Revenue_per_Voyage_USD']).sum())"),
    dict(id="Q19", difficulty="hard", kind="number",
         question="Of the voyages in Rough weather, what percentage had a turnaround time above the median turnaround time of the whole dataset?",
         solution="overall = df['Turnaround_Time_hours'].median()\nrough = df[df['Weather_Condition'] == 'Rough']\nanswer = float((rough['Turnaround_Time_hours'] > overall).mean() * 100)"),
    dict(id="Q20", difficulty="hard", kind="text",
         question="Which calendar quarter had the highest average load percentage? Give the answer as text in the form YYYYQn, for example 2023Q3.",
         solution="q = df.groupby(df['Date'].dt.to_period('Q'))['Average_Load_Percentage'].mean()\nanswer = str(q.idxmax())"),
]
print(len(QUESTIONS), "questions:", pd.Series([q["difficulty"] for q in QUESTIONS]).value_counts().to_dict())

## 4. Running model code safely

The models return Python code as text. Running code written by an AI model is a risk, so the executor below:

- gives the code a **copy** of the data, so it cannot change the original,
- allows only a short list of built-in functions,
- blocks every import except `pandas`, `numpy`, `math` and `datetime`,
- catches any error and reports it instead of crashing the run.

Limit: it does not stop an infinite loop. For untrusted models at scale, a separate process with a timeout would be needed.

In [ ]:
ALLOWED_IMPORTS = {"pandas", "numpy", "math", "datetime"}

def _safe_import(name, *args, **kwargs):
    if name.split(".")[0] not in ALLOWED_IMPORTS:
        raise ImportError(f"import of '{name}' is blocked")
    return __import__(name, *args, **kwargs)

SAFE_BUILTINS = {n: getattr(builtins, n) for n in [
    "abs", "all", "any", "bool", "dict", "enumerate", "filter", "float", "int", "isinstance",
    "len", "list", "map", "max", "min", "print", "range", "reversed", "round", "set", "sorted",
    "str", "sum", "tuple", "zip"]}
SAFE_BUILTINS["__import__"] = _safe_import


def run_code(source):
    """Run code text. Return (answer, error_message)."""
    env = {"__builtins__": SAFE_BUILTINS, "pd": pd, "np": np, "df": df.copy()}
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            exec(source, env)
    except Exception as e:
        return None, f"{type(e).__name__}: {e}"
    if "answer" not in env:
        return None, "NoAnswer: the code never set a variable named `answer`"
    return env["answer"], None


def extract_code(response):
    """Pull the Python code out of a model reply."""
    blocks = re.findall(r"```(?:python|py)?\s*\n(.*?)```", response, flags=re.S)
    return (blocks[0] if blocks else response).strip()

## 5. The grader

Grading rules, fixed before any model was run:

| Answer type | Counted correct when |
|---|---|
| number | within 0.5% of the reference (or within 0.005 for values near zero) |
| text | same label, ignoring letter case and surrounding spaces |
| list | same labels in the same order |

Every response gets one outcome: `correct`, `wrong_answer`, `runtime_error` (the code crashed) or `no_answer` (the code ran but produced nothing to grade).

In [ ]:
def normalise(x):
    """Turn Pandas and NumPy objects into plain Python values."""
    if isinstance(x, pd.DataFrame):
        x = x.iloc[:, 0] if x.shape[1] == 1 else x.values
    if isinstance(x, (pd.Series, pd.Index, np.ndarray)):
        x = x.tolist()
    if isinstance(x, tuple):
        x = list(x)
    if isinstance(x, list):
        x = [normalise(v) for v in x]
        return x[0] if len(x) == 1 else x
    if isinstance(x, np.generic):
        x = x.item()
    if isinstance(x, pd.Period):
        x = str(x)
    return x


def is_correct(pred, ref, kind):
    pred, ref = normalise(pred), normalise(ref)
    try:
        if kind == "number":
            return math.isclose(float(pred), float(ref), rel_tol=0.005, abs_tol=0.005)
        if kind == "text":
            if isinstance(pred, list):
                pred = " | ".join(str(p) for p in pred)
            return str(pred).strip().lower() == str(ref).strip().lower()
        if kind == "list":
            if not isinstance(pred, list):
                return False
            return [str(p).strip().lower() for p in pred] == [str(r).strip().lower() for r in ref]
    except (TypeError, ValueError):
        return False
    return False


def grade(response, q):
    source = extract_code(response)
    pred, error = run_code(source)
    if error and error.startswith("NoAnswer"):
        status = "no_answer"
    elif error:
        status = "runtime_error"
    elif is_correct(pred, q["reference"], q["kind"]):
        status = "correct"
    else:
        status = "wrong_answer"
    return dict(code=source, prediction=repr(normalise(pred))[:120], status=status, error=error or "")

### Check the grader before trusting it

Two tests. The reference solutions must all run and score 100%. Deliberately broken answers must all be rejected. If either test fails, the benchmark is wrong, not the model.

In [ ]:
for q in QUESTIONS:
    q["reference"], err = run_code(q["solution"])
    assert err is None, f"{q['id']} reference solution failed: {err}"

self_check = [grade(f"```python\n{q['solution']}\n```", q)["status"] for q in QUESTIONS]
assert all(s == "correct" for s in self_check), "grader rejects a reference solution"

assert not is_correct(101, 100, "number")                       # 1% off: must fail
assert is_correct(100.2, 100, "number")                         # 0.2% off: must pass
assert is_correct(" tanker ", "Tanker", "text")
assert not is_correct(["a", "b"], ["b", "a"], "list")           # order matters
assert grade("answer = df['No_Such_Column'].sum()", QUESTIONS[0])["status"] == "runtime_error"
assert grade("x = 1", QUESTIONS[0])["status"] == "no_answer"
assert grade("import os\nanswer = 1", QUESTIONS[0])["status"] == "runtime_error"   # blocked import
print("Grader checks passed.")

pd.DataFrame([{"id": q["id"], "difficulty": q["difficulty"], "question": q["question"],
               "ground truth": normalise(q["reference"])} for q in QUESTIONS])

## 6. Two prompts

The same question is sent under two prompts, to measure how much prompt quality matters.

- **Baseline:** column names and the question. This is what a hurried user would write.
- **Improved:** adds data types, the allowed category labels, the meaning of `Unknown`, and exact rules for the output.

Both prompts ask for the result in a variable named `answer`. Without that, the comparison would only measure formatting.

In [ ]:
BASELINE_SYSTEM = "You are a data analyst. Write Python pandas code to answer the question."

def baseline_user(question):
    return (f"A pandas DataFrame named df has these columns: {', '.join(df.columns)}.\n\n"
            f"Question: {question}\n\nStore the final result in a variable named answer.")


def schema_text():
    lines = []
    for col in df.columns:
        if col in CATEGORICAL:
            lines.append(f"- {col} (text). Allowed values: {sorted(df[col].unique())}")
        elif col == "Date":
            lines.append(f"- {col} (datetime64), weekly, {df[col].min():%Y-%m-%d} to {df[col].max():%Y-%m-%d}")
        else:
            lines.append(f"- {col} ({df[col].dtype}), range {df[col].min():,.2f} to {df[col].max():,.2f}")
    return "\n".join(lines)

IMPROVED_SYSTEM = f"""You are a careful data analyst. You answer questions about ship voyage records by writing Python pandas code.

The data is already loaded in a pandas DataFrame named df. Each row is one voyage record. Columns:
{schema_text()}

Rules:
1. Do not read files or create new data. Use df only, with pandas as pd and numpy as np.
2. Use column names and category labels exactly as written above. They are case-sensitive.
3. The label 'Unknown' means the value is missing. Keep those rows unless the question says to exclude them.
4. Store the final result in a variable named answer.
5. A numeric answer must be a plain number, not rounded. A label answer must be a string. A ranking must be a Python list of strings.
6. Reply with one ```python code block and nothing else."""

def improved_user(question):
    return f"Question: {question}"


PROMPTS = {
    "baseline": lambda q: [{"role": "system", "content": BASELINE_SYSTEM},
                           {"role": "user", "content": baseline_user(q)}],
    "improved": lambda q: [{"role": "system", "content": IMPROVED_SYSTEM},
                           {"role": "user", "content": improved_user(q)}],
}
print(IMPROVED_SYSTEM)

## 7. Run the models

Each model is loaded once, answers all 20 questions under both prompts, then is removed from memory. Decoding is greedy (`do_sample=False`), so the run is repeatable.

In [ ]:
def simulated_reply(model_name, prompt_name, q):
    """DRY_RUN only. A fake model that is sometimes right and sometimes wrong in realistic ways."""
    rnd = random.Random(f"{model_name}|{prompt_name}|{q['id']}")
    skill = (0.65 if "1.5B" in model_name else 0.45) + (0.15 if prompt_name == "improved" else 0.0)
    skill -= {"easy": 0.0, "medium": 0.10, "hard": 0.25}[q["difficulty"]]
    src = q["solution"]
    if rnd.random() >= skill:
        fault = rnd.choice(["column", "noanswer", "aggregate"])
        if fault == "column":
            src = src.replace("_USD", "").replace("_nm", "", 1)
        elif fault == "noanswer":
            src = src.replace("answer =", "result =")
        else:
            src = src.replace(".mean()", ".max()").replace(".sum()", ".mean()", 1)
    return f"```python\n{src}\n```"


def load_model(model_id):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype="auto", device_map="auto")
    return tok, model


def real_reply(tok, model, messages):
    import torch
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=300, do_sample=False)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

In [ ]:
rows = []
for model_name, model_id in MODELS.items():
    if not DRY_RUN:
        tok, model = load_model(model_id)
    for prompt_name, build_messages in PROMPTS.items():
        for q in QUESTIONS:
            if DRY_RUN:
                response = simulated_reply(model_name, prompt_name, q)
            else:
                response = real_reply(tok, model, build_messages(q["question"]))
            rows.append(dict(model=model_name, prompt=prompt_name, id=q["id"], difficulty=q["difficulty"],
                             question=q["question"], response=response,
                             reference=repr(normalise(q["reference"]))[:120], **grade(response, q)))
        done = pd.DataFrame(rows).query("model == @model_name and prompt == @prompt_name")
        print(f"{model_name:22s} {prompt_name:9s} {(done['status'] == 'correct').sum():2d}/{len(done)} correct")
    if not DRY_RUN:
        import gc, torch
        del tok, model
        gc.collect()
        torch.cuda.empty_cache()

results = pd.DataFrame(rows)
results.to_csv("eval_results.csv", index=False)
if DRY_RUN:
    print("\nDRY RUN: these numbers come from a simulated model and are not real results.")

## 8. Results

### 8.1 Accuracy by model and prompt

In [ ]:
results["is_correct"] = results["status"] == "correct"
accuracy = (results.pivot_table(index="model", columns="prompt", values="is_correct", aggfunc="mean") * 100).round(1)
accuracy = accuracy.loc[list(MODELS), list(PROMPTS)]
accuracy

In [ ]:
INK, MUTED, GRID = "#0b0b0b", "#898781", "#e1e0d9"
SERIES = {"baseline": "#2a78d6", "improved": "#eb6834"}

fig, ax = plt.subplots(figsize=(7.5, 4.2), facecolor="#fcfcfb")
ax.set_facecolor("#fcfcfb")
x = np.arange(len(accuracy))
width = 0.34
for i, prompt_name in enumerate(accuracy.columns):
    bars = ax.bar(x + (i - 0.5) * (width + 0.02), accuracy[prompt_name], width,
                  color=SERIES[prompt_name], label=f"{prompt_name.capitalize()} prompt")
    ax.bar_label(bars, fmt="%.0f%%", padding=3, color=INK, fontsize=10)

ax.set_xticks(x, accuracy.index, color=INK)
ax.set_ylim(0, 100)
ax.set_yticks([0, 25, 50, 75, 100], ["0%", "25%", "50%", "75%", "100%"], color=MUTED)
ax.set_ylabel("Questions answered correctly", color=MUTED)
ax.yaxis.grid(True, color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
for side in ["top", "right", "left"]:
    ax.spines[side].set_visible(False)
ax.spines["bottom"].set_color("#c3c2b7")
ax.tick_params(length=0)
title = "Accuracy on 20 maritime data questions"
if DRY_RUN:
    title += "  (DRY RUN: simulated, not real)"
ax.set_title(title, loc="left", color=INK, fontsize=12, pad=38)
ax.legend(frameon=False, loc="lower left", ncols=2, bbox_to_anchor=(-0.01, 1.0), labelcolor=INK)
plt.tight_layout()
plt.savefig("accuracy_by_model_and_prompt.png", dpi=160)
plt.show()

### 8.2 Accuracy by difficulty

In [ ]:
by_difficulty = (results.pivot_table(index=["model", "prompt"], columns="difficulty",
                                     values="is_correct", aggfunc="mean") * 100).round(0)
by_difficulty[["easy", "medium", "hard"]]

### 8.3 How the models fail

A wrong answer that runs cleanly is the most dangerous outcome, because nothing warns the user. A crash is safer: it is visible.

In [ ]:
outcomes = (results.groupby(["model", "prompt"])["status"].value_counts().unstack(fill_value=0)
            .reindex(columns=["correct", "wrong_answer", "runtime_error", "no_answer"], fill_value=0))
outcomes

In [ ]:
errors = results[results["status"] == "runtime_error"].copy()
errors["error_type"] = errors["error"].str.split(":").str[0]
errors.groupby(["model", "prompt"])["error_type"].value_counts().unstack(fill_value=0)

### 8.4 Ranking the two models head to head

For each question and prompt, the two models are compared directly. A model wins a question when it is correct and the other is not.

In [ ]:
a_name, b_name = list(MODELS)
wide = results.pivot_table(index=["prompt", "id"], columns="model", values="is_correct", aggfunc="first")
head_to_head = pd.DataFrame({
    f"{b_name} wins": (wide[b_name] & ~wide[a_name]).groupby("prompt").sum(),
    f"{a_name} wins": (wide[a_name] & ~wide[b_name]).groupby("prompt").sum(),
    "both correct": (wide[a_name] & wide[b_name]).groupby("prompt").sum(),
    "both wrong": (~wide[a_name] & ~wide[b_name]).groupby("prompt").sum(),
})
head_to_head

In [ ]:
hardest = (results.groupby(["id", "difficulty"])["is_correct"].mean() * 100).round(0).sort_values().head(5)
print("Five questions the models found hardest (percent of the four runs that got it right):")
hardest.to_frame("percent correct").join(pd.DataFrame(QUESTIONS).set_index("id")["question"])

## 9. Human review of failures

Automatic grading says **that** a response failed. It does not say **why**. This step exports every failed response so that each one gets a cause and a written rationale from a human reviewer.

**Rubric for `error_category`**

| Category | Meaning |
|---|---|
| `wrong_column` | used a column that does not exist, or the wrong one |
| `wrong_filter` | ignored or misapplied a condition, such as "exclude Unknown" |
| `wrong_aggregation` | wrong maths, such as mean of ratios when the question asked for ratio of totals |
| `wrong_format` | right value, wrong shape or type |
| `invalid_code` | syntax error, blocked import, or code unrelated to the question |
| `no_answer` | never set `answer` |

**Rationale standard:** one or two sentences naming the exact line that is wrong, what it does, and what the correct logic is.

In [ ]:
review = results[results["status"] != "correct"][
    ["model", "prompt", "id", "difficulty", "question", "code", "prediction", "reference", "status", "error"]].copy()
review["error_category"] = ""     # fill in by hand, using the rubric above
review["rationale"] = ""          # fill in by hand
review.to_csv("human_review.csv", index=False)
print(f"{len(review)} failed responses exported to human_review.csv for review")
review.head(3)

### Worked reviews

> **To complete after the real run.** Pick the three most instructive failures from `human_review.csv`. For each, paste the model's code, name the error category and write the rationale. This section is the evidence of reviewing skill, so it should be written by hand.

**Review 1**: *question id, model, prompt*

- Model code:
- Error category:
- Rationale:

**Review 2**

**Review 3**

## 10. Fine-tuning dataset (SFT)

The 20 verified solutions are exported in the chat format used for supervised fine-tuning: a system message, the user's question and the ideal assistant reply. Each record was produced by code that ran and matched the ground truth, so the dataset contains no unverified answers.

Four questions are held out for validation, spread across the difficulty levels, so a fine-tuned model could be tested on questions it never saw.

In [ ]:
def sft_record(q):
    return {"messages": [
        {"role": "system", "content": IMPROVED_SYSTEM},
        {"role": "user", "content": improved_user(q["question"])},
        {"role": "assistant", "content": f"```python\n{q['solution']}\n```"},
    ]}

VALIDATION_IDS = {"Q05", "Q10", "Q14", "Q19"}
train = [sft_record(q) for q in QUESTIONS if q["id"] not in VALIDATION_IDS]
val = [sft_record(q) for q in QUESTIONS if q["id"] in VALIDATION_IDS]

for name, records in [("sft_train.jsonl", train), ("sft_val.jsonl", val)]:
    with open(name, "w") as f:
        for r in records:
            f.write(json.dumps(r) + "\n")
    print(f"{name}: {len(records)} records")

print("\nExample record (assistant turn):\n" + train[0]["messages"][2]["content"])

## 11. Findings

> **To complete after the real run**, using the tables above. Replace each bracket with the number from this notebook.

- **Best configuration:** [model] with the [prompt] prompt answered [n]/20 correctly.
- **Effect of the prompt:** the improved prompt changed accuracy from [x]% to [y]% on the smaller model and from [x]% to [y]% on the larger one.
- **Effect of difficulty:** accuracy fell from [x]% on easy questions to [y]% on hard ones.
- **Most common failure:** [category], which the improved prompt [did / did not] reduce.
- **Silent errors:** [n] responses ran without error but gave the wrong answer.

### Limitations

- 20 questions is a small benchmark. One question is worth 5 points, so small gaps between models are not meaningful.
- All questions use one dataset. Results may not carry over to other tables.
- The dataset is synthetic, so the business findings in the ground truth are for demonstration only.
- Each question is run once with greedy decoding. Sampling several times would show how stable each model is.

### Next steps

- Grow the benchmark to 100+ questions across three datasets.
- Fine-tune the smaller model on `sft_train.jsonl` and re-run this evaluation to measure the gain.
- Add a larger hosted model as a third comparison point.